# Course Verifier — Google Colab Runner
Run your heavy verifier on Colab for free. Chrome + 12 GB RAM included.

**Steps before you start:**
1. Upload `link_compile.pdf` to the Files panel (left sidebar)
2. Upload your `.env` file to the Files panel
3. Run cells top to bottom

---

## Cell 1: Mount Google Drive (optional but recommended)
This lets you save results permanently in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Mounted Google Drive")

## Cell 2: Clone repo & install dependencies
Takes ~3 minutes. Installs Chrome, Tesseract OCR, Python libs.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Shlok-Parekh09/course-verifier.git"
BRANCH   = "yug-render-deploy"
WORKDIR  = "/content/course-verifier"

# Clone if missing, otherwise hard-reset to latest remote so code fixes take
# effect. We use reset --hard (not pull) because later cells patch the source
# in-place (NUM_BROWSERS), which would otherwise make a pull fail with conflicts.
if not os.path.exists(WORKDIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, WORKDIR], check=True)
    print("Cloned to", WORKDIR)
else:
    os.chdir(WORKDIR)
    subprocess.run(['git', 'fetch', 'origin'], check=False)
    subprocess.run(['git', 'checkout', BRANCH], check=False)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=False)
    print("Reset to latest origin/", BRANCH)
os.chdir(WORKDIR)
print("HEAD:", subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout.strip())

# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq wget gnupg2 ca-certificates fonts-liberation libglib2.0-0 libnss3 libgconf-2-4 libfontconfig1 libxss1 libappindicator3-1 libatk-bridge2.0-0 libgtk-3-0 libxcomposite1 libxcursor1 libxdamage1 libxi6 libxtst6 libxrandr2 libasound2 libpangocairo-1.0-0 libatspi2.0-0 libcups2 libdrm2 libgbm1 libxkbcommon0 tesseract-ocr tesseract-ocr-eng unzip curl

# Install Google Chrome Stable
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add - 2>/dev/null
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google.list
!apt-get update -qq && apt-get install -y -qq google-chrome-stable

# Show the installed Chrome version (the verifier now auto-matches ChromeDriver to this)
!google-chrome --version

# Install Python dependencies
!pip install -q -r requirements.txt
!playwright install chromium || true

# Purge any stale cached ChromeDriver so undetected_chromedriver fetches the right one.
!rm -rf ~/.local/share/undetected_chromedriver /tmp/undetected* 2>/dev/null || true

print("Setup complete!")

## Cell 3: Copy uploaded files into workspace
Make sure you uploaded `link_compile.pdf` and `.env` via the Files panel first.

In [ ]:
import shutil, os

# Copy from default upload root to repo workspace
upload_root = "/content"
for fname in ['link_compile.pdf', '.env']:
    src = os.path.join(upload_root, fname)
    dst = os.path.join(WORKDIR, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"Copied {fname}")

assert os.path.exists("link_compile.pdf"), "Upload link_compile.pdf via the Files panel!"
assert os.path.exists(".env"), "Upload .env via the Files panel!"
print("PDF and .env detected. Ready to run.")

## Cell 4: Run the verifier
Adjust `START_PAGE` and `END_PAGE` below. Use `--no-email` here; we send it separately in the next cell.

In [ ]:
import os

START_PAGE = 602
END_PAGE   = 1890
BROWSERS   = 2

os.environ["VERIFIER_NO_FORCE_EXIT"] = "true"

# Patch browser count in verifier source
with open("autonomous_course_verifier.py", "r", encoding="utf-8") as f:
    src = f.read()
src = src.replace("NUM_BROWSERS = 6", f"NUM_BROWSERS = {BROWSERS}")
with open("autonomous_course_verifier.py", "w", encoding="utf-8") as f:
    f.write(src)

!python run_verifier_pages.py link_compile.pdf --pages {START_PAGE} {END_PAGE} --no-email

print("Verifier finished!")

## Cell 5: Email the report
Requires your `.env` to have SMTP credentials. If email fails, check the error below.

In [ ]:
import glob, os
from email_sender import send_report

pdfs = glob.glob("Verification_Report_Pages_*.pdf")
if pdfs:
    pdf = pdfs[0]
    ok, msg = send_report(
        f"Course Verification Complete — Pages {START_PAGE}-{END_PAGE}",
        f"Colab run finished.\nPage range: {START_PAGE} – {END_PAGE}\nReport: {os.path.basename(pdf)}",
        pdf
    )
    print(f"[{'OK' if ok else 'FAIL'}] Email: {msg}")
else:
    print("No PDF report found. Check for errors above.")

## Cell 6: Save results to Google Drive (optional)
Copies PDF, Excel, and JSON checkpoint into your Drive under `verifier_output/YYYYMMDD_HHMMSS/`.

In [ ]:
import shutil, datetime, glob, os

outdir = f"/content/drive/MyDrive/verifier_output/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(outdir, exist_ok=True)

for pattern in ["*.xlsx", "Verification_Report_Pages_*.pdf", "autonomous_verified_link_compile.pdf.json"]:
    for f in glob.glob(pattern):
        if os.path.exists(f):
            shutil.copy(f, outdir)
            print(f"Saved: {f}")

print("Done! Check your Google Drive under verifier_output/")